<a href="https://colab.research.google.com/github/Jugrankrish/Chisel-AI/blob/main/chisel_ai_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chisel-AI — Google Colab Runner
**Text-guided 3D Gaussian Splatting object removal pipeline**

> Run cells **top-to-bottom**. Cells 0–7 are one-time setup (fast on repeat sessions because weights live on Drive). Cell 7 must run each session (secrets). Cells 8–9 run the actual pipeline.

**Before starting:**
- Runtime → Change runtime type → **T4 GPU**
- Upload assets to `MyDrive/Chisel-AI/` (see migration plan for folder layout)
- Add `OPENROUTER_API_KEY` to 🔑 Colab Secrets (left panel)

In [1]:
# ═══ CELL 0: GPU / Runtime Check ═══════════════════════════════════════════════
import subprocess, sys
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                   capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError('No GPU found! Runtime → Change runtime type → T4 GPU')
print('GPU:', r.stdout.strip())

import torch
print(f'torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}')

GPU: Tesla T4, 15360 MiB
torch 2.2.2+cu118 | CUDA available: True


In [2]:
# ═══ CELL 1: Mount Google Drive ════════════════════════════════════════════════
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT    = Path('/content/drive/MyDrive/Chisel-AI')
DRIVE_WEIGHTS = DRIVE_ROOT / 'weights'
DRIVE_DATASET = DRIVE_ROOT / 'datasets' / 'tandt' / 'truck'
DRIVE_PLY_IN  = DRIVE_ROOT / 'point_clouds' / 'truck_iteration_30000.ply'
DRIVE_OUT     = DRIVE_ROOT / 'outputs'
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

for p in [DRIVE_WEIGHTS, DRIVE_DATASET / 'images',
          DRIVE_DATASET / 'sparse' / '0', DRIVE_PLY_IN]:
    status = '✅' if p.exists() else '❌ MISSING'
    print(f'{status}  {p}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅  /content/drive/MyDrive/Chisel-AI/weights
✅  /content/drive/MyDrive/Chisel-AI/datasets/tandt/truck/images
✅  /content/drive/MyDrive/Chisel-AI/datasets/tandt/truck/sparse/0
✅  /content/drive/MyDrive/Chisel-AI/point_clouds/truck_iteration_30000.ply


In [3]:
# ═══ CELL 2: Clone Repo ════════════════════════════════════════════════════════
import subprocess, os, sys
REPO_DIR = '/content/chisel-ai'

if not os.path.exists(f'{REPO_DIR}/text_removal_pipeline/pipeline.py'):
    print('Cloning Chisel-AI ...')
    subprocess.run(['git','clone','https://github.com/Jugrankrish/Chisel-AI.git', REPO_DIR], check=True)
    print('Cloned.')
else:
    print('Already cloned — pulling latest ...')
    subprocess.run(['git','-C', REPO_DIR,'pull'], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('Repo root:', REPO_DIR)

Already cloned — pulling latest ...
Repo root: /content/chisel-ai


In [4]:
# ═══ CELL 3: Install Pinned torch 2.1.2+cu118 ══════════════════════════════════
#
# REASON: local nerfstudio env = torch 2.1.2+cu118 (CUDA 11.8).
#         Colab default = torch 2.x+cu12x. groundingdino-py 0.4.0 must
#         link against the SAME ABI or you get illegal-memory CUDA errors.
#         T4 supports both cu118 and cu12x — installing cu118 is safe.
#
# AFTER THIS CELL: if torch is reinstalled, kernel restart is required.
#                  Then resume from CELL 4.

import subprocess, torch
print('Current torch:', torch.__version__)

if 'cu118' not in torch.__version__ or '2.1.2' not in torch.__version__:
    print('Reinstalling torch 2.1.2+cu118 ...')
    subprocess.run(['pip','install','--quiet','--upgrade',
                    'torch==2.2.2+cu118','torchvision==0.17.2+cu118',
                    '--index-url','https://download.pytorch.org/whl/cu118'], check=True)
    print('Done. ⚠  RESTART KERNEL NOW: Runtime → Restart session, then run from Cell 4.')
else:
    print('torch 2.1.2+cu118 already installed — no restart needed, continue to Cell 4.')

Current torch: 2.2.2+cu118
Reinstalling torch 2.1.2+cu118 ...
Done. ⚠  RESTART KERNEL NOW: Runtime → Restart session, then run from Cell 4.


In [14]:
# ═══ CELL 4: Install All Dependencies ══════════════════════════════════════════
# Run AFTER kernel restart (if Cell 3 reinstalled torch).
# Install order is important: groundingdino first (links to torch).

import subprocess

def pip(*args, **kw):
    check = kw.get('check', True)
    cmd = ['pip','install','--quiet'] + list(args)
    print(' '.join(['pip install'] + list(args)), '...')
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0 and check:
        print('  STDERR:', r.stderr[-800:])
        raise RuntimeError('pip failed: ' + ' '.join(args))
    print('  done')
    return r

# 1. GroundingDINO (must come first after torch)
r = pip('groundingdino-py==0.4.0', check=False)
if r.returncode != 0:
    print('  Wheel not found, building from source ...')
    subprocess.run(['bash','-c',
        'git clone https://github.com/IDEA-Research/GroundingDINO.git /tmp/gdino '
        '&& pip install --quiet -e /tmp/gdino --no-build-isolation'], check=True)
    print('  GroundingDINO built from source.')

# 2. SAM
pip('git+https://github.com/facebookresearch/segment-anything.git')

# 3. Supporting packages (version-pinned to match nerfstudio env)
pip('opencv-python==4.9.0.80')
pip('plyfile==1.1.3')
pip('scikit-learn==1.7.2')
pip('tqdm==4.70.0')
pip('numpy==1.26.4')
pip('timm==0.6.7')

# pycolmap: try 4.1.1, fall back to 3.13.0
r = pip('pycolmap==4.1.1', check=False)
if r.returncode != 0:
    print('  pycolmap 4.1.1 wheel missing, trying 3.13.0 ...')
    pip('pycolmap==3.13.0')

# 3b. Pin transformers — newer versions require torch>=2.4 and silently
#     disable PyTorch support, which breaks GroundingDINO's BertModel load.
pip('transformers==4.44.2')

# 4. LLM clients
pip('openai==2.50.0')
pip('google-genai==2.15.0')
pip('boto3==1.43.59')

# Smoke test
import torch
from groundingdino.util.inference import load_model
from segment_anything import sam_model_registry
print(f'\ntorch {torch.__version__} | CUDA: {torch.cuda.is_available()} | GDino + SAM importable')
print('All dependencies installed.')

pip install groundingdino-py==0.4.0 ...
  done
pip install git+https://github.com/facebookresearch/segment-anything.git ...
  done
pip install opencv-python==4.9.0.80 ...
  done
pip install plyfile==1.1.3 ...
  done
pip install scikit-learn==1.7.2 ...
  done
pip install tqdm==4.70.0 ...
  done
pip install numpy==1.26.4 ...
  done
pip install timm==0.6.7 ...
  done
pip install pycolmap==4.1.1 ...
  done
pip install transformers==4.44.2 ...
  done
pip install openai==2.50.0 ...
  done
pip install google-genai==2.15.0 ...
  done
pip install boto3==1.43.59 ...
  done

torch 2.2.2+cu118 | CUDA: True | GDino + SAM importable
All dependencies installed.


In [15]:
# ═══ CELL 5: Verify/Download Weights to Drive ═══════════════════════════════════
# Weights live on Drive permanently. This cell downloads them once if missing.
# On subsequent sessions it just confirms they exist (fast).

import os, urllib.request
from pathlib import Path

DRIVE_WEIGHTS = Path('/content/drive/MyDrive/Chisel-AI/weights')

DOWNLOADS = {
    'groundingdino_swint_ogc.pth': (
        'https://github.com/IDEA-Research/GroundingDINO/releases/download/'
        'v0.1.0-alpha/groundingdino_swint_ogc.pth'
    ),
    'GroundingDINO_SwinT_OGC.py': (
        'https://raw.githubusercontent.com/IDEA-Research/GroundingDINO/'
        'main/groundingdino/config/GroundingDINO_SwinT_OGC.py'
    ),
    'sam_vit_b_01ec64.pth': (
        'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'
    ),
}

def download_with_progress(url, dest):
    dest.parent.mkdir(parents=True, exist_ok=True)
    print(f'  Downloading {dest.name} to Drive ...')
    def hook(c, b, t): print(f'\r    {min(c*b/t*100,100):.1f}%', end='', flush=True) if t > 0 else None
    urllib.request.urlretrieve(url, str(dest), hook)
    print(f'\n  Saved: {dest.stat().st_size/1e6:.0f} MB')

for fname, url in DOWNLOADS.items():
    dest = DRIVE_WEIGHTS / fname
    if dest.exists():
        print(f'  OK: {fname} ({dest.stat().st_size/1e6:.0f} MB) already on Drive')
    else:
        download_with_progress(url, dest)

# Symlink weights into the repo locations the code expects
REPO_DIR = '/content/chisel-ai'
repo_w = Path(REPO_DIR) / 'text_removal_pipeline' / 'weights'
repo_w.mkdir(parents=True, exist_ok=True)

for f in ['groundingdino_swint_ogc.pth', 'GroundingDINO_SwinT_OGC.py']:
    dst = repo_w / f
    if dst.exists() or dst.is_symlink(): dst.unlink()
    os.symlink(str(DRIVE_WEIGHTS / f), str(dst))
    print(f'  Symlinked {f} -> Drive')

print('Weights ready.')

  OK: groundingdino_swint_ogc.pth (694 MB) already on Drive
  OK: GroundingDINO_SwinT_OGC.py (0 MB) already on Drive
  OK: sam_vit_b_01ec64.pth (375 MB) already on Drive
  Symlinked groundingdino_swint_ogc.pth -> Drive
  Symlinked GroundingDINO_SwinT_OGC.py -> Drive
Weights ready.


In [16]:
# ═══ CELL 6: Copy Dataset to /content/ (fast local I/O) ═══════════════════════
# Drive FUSE adds ~3-5x latency iterating 251 images. Copy once per session.

import shutil, time
from pathlib import Path

DRIVE_DATASET  = Path('/content/drive/MyDrive/Chisel-AI/datasets/tandt/truck')
LOCAL_DATASET  = Path('/content/dataset/tandt/truck')
DRIVE_PLY_IN = Path('/content/drive/MyDrive/Chisel-AI/point_clouds/truck_iteration_30000.ply/point_cloud.ply')
LOCAL_PLY_IN   = Path('/content/point_clouds/truck_iteration_30000.ply')

t0 = time.time()

for subdir in ['images', 'sparse']:
    dst = LOCAL_DATASET / subdir
    if not dst.exists():
        print(f'Copying {subdir}/ from Drive ...')
        shutil.copytree(str(DRIVE_DATASET / subdir), str(dst))
        print(f'  Done.')
    else:
        print(f'  {subdir}/ already in /content/ (skip)')

if not LOCAL_PLY_IN.exists():
    LOCAL_PLY_IN.parent.mkdir(parents=True, exist_ok=True)
    print('Copying input PLY (~487 MB) from Drive ...')
    shutil.copy2(str(DRIVE_PLY_IN), str(LOCAL_PLY_IN))
    print(f'  Done ({LOCAL_PLY_IN.stat().st_size/1e6:.0f} MB)')
else:
    print(f'  Input PLY already in /content/ ({LOCAL_PLY_IN.stat().st_size/1e6:.0f} MB)')

n_imgs = len(list((LOCAL_DATASET / 'images').glob('*')))
print(f'Dataset ready: {n_imgs} images  ({time.time()-t0:.0f}s total)')

  images/ already in /content/ (skip)
  sparse/ already in /content/ (skip)
  Input PLY already in /content/ (510 MB)
Dataset ready: 251 images  (0s total)


In [17]:
# ═══ CELL 7: Load API Keys from Colab Secrets ══════════════════════════════════
# Add secrets in Colab: left panel 🔑 Secrets → + Add new secret
#
# Secrets to add:
#   OPENROUTER_API_KEY  ← your sk-or-v1-... value
#   GEMINI_API_KEY      ← if you have one
#   OPENAI_API_KEY      ← if you have one
#   AWS_ACCESS_KEY_ID   ← if using Bedrock
#   AWS_SECRET_ACCESS_KEY
#   AWS_DEFAULT_REGION
#
# llm_agent.py reads from os.environ -- no code changes needed.
# Use --llm none for offline test with zero API keys.

import os
from google.colab import userdata

SECRETS = ['OPENROUTER_API_KEY','GEMINI_API_KEY','OPENAI_API_KEY',
           'AWS_ACCESS_KEY_ID','AWS_SECRET_ACCESS_KEY','AWS_DEFAULT_REGION']

loaded, missing = [], []
for name in SECRETS:
    try:
        val = userdata.get(name)
        if val: os.environ[name] = val; loaded.append(name)
        else: missing.append(name)
    except: missing.append(name)

os.environ.setdefault('GEMINI_MODEL', 'gemini-2.0-flash')
print('Loaded:', loaded)
print('Not set (optional):', missing)

Loaded: ['OPENROUTER_API_KEY']
Not set (optional): ['GEMINI_API_KEY', 'OPENAI_API_KEY', 'AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY', 'AWS_DEFAULT_REGION']


In [18]:
# ═══ CELL 8: Configure Pipeline ════════════════════════════════════════════════
from pathlib import Path

REPO_DIR = '/content/chisel-ai'

# ── Inputs ────────────────────────────────────────────────────────────────────
PLY_INPUT     = '/content/point_clouds/truck_iteration_30000.ply'
IMAGES_DIR    = '/content/dataset/tandt/truck/images'
COLMAP_DIR    = '/content/dataset/tandt/truck/sparse/0'
SAM_CKPT      = '/content/drive/MyDrive/Chisel-AI/weights/sam_vit_b_01ec64.pth'

# ── Outputs ───────────────────────────────────────────────────────────────────
MASKS_DIR     = f'{REPO_DIR}/text_removal_pipeline/masks'
OUTPUT_PLY    = f'{REPO_DIR}/text_removal_pipeline/outputs/cleaned.ply'

# ── Pipeline options (edit these) ─────────────────────────────────────────────
REMOVAL_TEXT  = 'remove the truck'  # ← change for different objects
LLM_BACKEND   = 'agent'             # 'agent'|'none'|'gemini'|'openai'|'ollama'
BOX_THRESHOLD = 0.30
TEXT_THRESH   = 0.25
RATIO         = 0.25
MAX_IMAGES    = None  # Set to e.g. 10 for a quick test; None = full dataset

# Verify inputs
for label, path in [('PLY', PLY_INPUT),('Images', IMAGES_DIR),
                     ('COLMAP', COLMAP_DIR),('SAM', SAM_CKPT)]:
    ok = Path(path).exists()
    print(f'{"OK" if ok else "MISSING"}: {label}: {path}')

print(f'\nTarget: "{REMOVAL_TEXT}"  |  LLM: {LLM_BACKEND}')

OK: PLY: /content/point_clouds/truck_iteration_30000.ply
OK: Images: /content/dataset/tandt/truck/images
OK: COLMAP: /content/dataset/tandt/truck/sparse/0
OK: SAM: /content/drive/MyDrive/Chisel-AI/weights/sam_vit_b_01ec64.pth

Target: "remove the truck"  |  LLM: agent


In [19]:
# ═══ CELL 9: Run Pipeline ══════════════════════════════════════════════════════
# Stages:
#   1. LLM prompt refinement  (instant with --llm none)
#   2. cameras.json from COLMAP
#   3. GroundingDINO + SAM segmentation  (~10-15 min for 251 images on T4)
#   4. Gaussian blacklist projection + PLY output

import subprocess, sys, os, time
from pathlib import Path

cmd = [
    sys.executable, '-m', 'text_removal_pipeline.pipeline',
    '--text',           REMOVAL_TEXT,
    '--ply',            PLY_INPUT,
    '--images',         IMAGES_DIR,
    '--colmap_dir',     COLMAP_DIR,
    '--sam',            SAM_CKPT,
    '--masks',          MASKS_DIR,
    '--output',         OUTPUT_PLY,
    '--llm',            LLM_BACKEND,
    '--box_threshold',  str(BOX_THRESHOLD),
    '--text_threshold', str(TEXT_THRESH),
    '--ratio',          str(RATIO),
]
if MAX_IMAGES is not None:
    cmd += ['--max_images', str(MAX_IMAGES)]

print('Command:', ' '.join(cmd))
print('=' * 65)

env = os.environ.copy()
env['PYTHONPATH'] = REPO_DIR

t0 = time.time()
proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()

elapsed = time.time() - t0
if proc.returncode != 0:
    print(f'Pipeline FAILED (exit {proc.returncode})')
elif Path(OUTPUT_PLY).exists():
    size_mb = Path(OUTPUT_PLY).stat().st_size / 1e6
    print(f'Pipeline complete in {elapsed:.0f}s. Output: {OUTPUT_PLY} ({size_mb:.0f} MB)')
else:
    print(f'Pipeline returned 0 but no output PLY found at {OUTPUT_PLY}')

Command: /usr/bin/python3 -m text_removal_pipeline.pipeline --text remove the truck --ply /content/point_clouds/truck_iteration_30000.ply --images /content/dataset/tandt/truck/images --colmap_dir /content/dataset/tandt/truck/sparse/0 --sam /content/drive/MyDrive/Chisel-AI/weights/sam_vit_b_01ec64.pth --masks /content/chisel-ai/text_removal_pipeline/masks --output /content/chisel-ai/text_removal_pipeline/outputs/cleaned.ply --llm agent --box_threshold 0.3 --text_threshold 0.25 --ratio 0.25
[OpenRouter] Request failed: Connection error.
[Bedrock] Request failed: Unable to locate credentials
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.

═════════════════════════════════════════════════════════════════
  TEXT-GUIDED 3D GAUSSIAN REMOVAL PIPELINE
══════════════════════════════════════════════════════════════

In [20]:
# ═══ CELL 10: Copy Output PLY to Google Drive ══════════════════════════════════
# /content/ is ephemeral. This saves the result permanently to Drive.

import shutil, time
from pathlib import Path

DRIVE_OUT = Path('/content/drive/MyDrive/Chisel-AI/outputs')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

src = Path(OUTPUT_PLY)
if not src.exists():
    print(f'No output PLY at {src}. Did Cell 9 succeed?')
else:
    ts  = time.strftime('%Y%m%d_%H%M%S')
    obj = REMOVAL_TEXT.lower().replace(' ','_').replace('remove_the_','')
    dst = DRIVE_OUT / f'cleaned_{obj}_{ts}.ply'

    print(f'Copying to Drive ...')
    shutil.copy2(str(src), str(dst))
    print(f'Saved: {dst}  ({dst.stat().st_size/1e6:.0f} MB)')
    print()
    print('View result:')
    print('  Supersplat (browser): https://playcanvas.com/supersplat/editor -> drag & drop')
    print('  MeshLab (desktop):    File -> Import Mesh -> cleaned.ply')
    print()
    print('Once verified, safe to delete locally (see migration plan section 6).')

Copying to Drive ...
Saved: /content/drive/MyDrive/Chisel-AI/outputs/cleaned_truck_20260804_133146.ply  (508 MB)

View result:
  Supersplat (browser): https://playcanvas.com/supersplat/editor -> drag & drop
  MeshLab (desktop):    File -> Import Mesh -> cleaned.ply

Once verified, safe to delete locally (see migration plan section 6).


---
## Troubleshooting

| Symptom | Fix |
|---|---|
| No GPU error (Cell 0) | Runtime → Change runtime type → T4 GPU |
| `illegal memory access` in GDino | Re-run Cell 3; restart kernel; resume from Cell 4 |
| `No masks generated` | Lower `BOX_THRESHOLD` to 0.20; simplify text (e.g. `'truck'` not `'red pickup truck'`) |
| `pycolmap` install fails | Pin to `pycolmap==3.13.0` in Cell 4 |
| Session expired mid-run | Set `MAX_IMAGES=50`, run in chunks; use `--skip_segmentation` to reuse masks |
| Output removes too much | Raise `RATIO` to 0.40 |
| Output removes too little | Lower `RATIO` to 0.15 |
| LLM backend fails | Set `LLM_BACKEND = 'none'` — works offline, no API key needed |